In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
from dotenv import load_dotenv
from google import genai

# Load variables from .env file
load_dotenv()

# Now the environment variable will be available
api_key = os.getenv("GOOGLE_API_KEY")

# Quick check
print("API Key found:", api_key is not None)

# Create client
client = genai.Client(api_key=api_key)

# Test a simple request
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Hello Gemini, write a short poem!"
)

print(response.text)


API Key found: True
A whisper of wind, a spark of light,
A quiet star to guide the night.
In every word and thought we share,
A bridge of minds is waiting there.


In [4]:
from langchain_community.document_loaders import PyPDFLoader


In [5]:
%pwd

'd:\\QGen\\notebook'

In [6]:
%cd ..

d:\QGen


In [7]:
file_path = "D:\QGen\data\SDGs_Booklet_Web_En.pdf"
loader = PyPDFLoader(file_path)
data= loader.load()
print(data[2].page_content[:300])

IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They kn


In [8]:
print(f"no. of pages={len(data)}")

no. of pages=24


In [9]:
question_gen = ""

for pages in data:
    question_gen += pages.page_content
    

In [10]:
question_gen

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

In [11]:
from langchain_text_splitters import TokenTextSplitter

splitter_ques_gen = TokenTextSplitter(
    chunk_size=10000,   # adjust depending on how big you want chunks
    chunk_overlap=200
)

chunks = splitter_ques_gen.split_documents(data)  # data = loader.load()
print(len(chunks))
print(chunks[0].page_content[:300])


21
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They kn


In [12]:
chunk_ques_gen = splitter_ques_gen.split_text(question_gen)

In [13]:
chunk_ques_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 ye

In [14]:
len(chunk_ques_gen)

1

In [15]:
from langchain_community.docstore.document import Document

In [16]:
doc_ques_gen = [Document(page_content = t) for t in chunk_ques_gen]

In [17]:
type(doc_ques_gen)


list

In [18]:
splitter_ans_gen = TokenTextSplitter(
    chunk_size=1000,   # adjust depending on how big you want chunks
    chunk_overlap=100
)


In [19]:
doc_ans_gen = splitter_ans_gen.split_documents(
    doc_ques_gen
)

doc_ans_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

In [20]:
len(doc_ans_gen)

5

In [21]:
!pip install -U langchain-google-genai

In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [23]:
llm_ques_gen_pipeline = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0.3
)

In [24]:
prompt_template = """
You are an expert at creating high-quality questions from educational materials and documentation.

Your goal is to help students, professionals, and learners thoroughly understand the given content by generating meaningful and relevant questions.

Study the following text carefully:

----------------
{text}
----------------

Instructions:
- Generate questions only from the provided text.
- Cover all the important concepts and key points.
- Do not omit any significant information.
- Avoid duplicate or repetitive questions.
- Make the questions clear, concise, and grammatically correct.
- Include a mix of factual, conceptual, and application-based questions whenever appropriate.
- Do not provide answers.
- Number each question.
- limit the questions to 20, only upto 20 questions are valid.
QUESTIONS:
"""

In [25]:
from langchain_core.prompts import PromptTemplate

In [26]:
PROMPT_QUESTIONS = PromptTemplate(template= prompt_template, input = ["text"])

In [27]:
refine_template = """
You are an expert at creating high-quality practice questions from educational material and documentation.

Your goal is to help students and learners prepare for exams, interviews, and assessments.

We have already generated some practice questions to a certain extent:
{existing_answer}

You now have the opportunity to refine the existing questions or add new ones using the additional context provided below.

----------------
{text}
----------------

Given the new context:
- Refine the existing questions if necessary.
- Add new questions for any important concepts that are not already covered.
- Do not remove good existing questions.
- Avoid duplicate or repetitive questions.
- Make sure no important information is missed.
- Generate questions only from the provided context.
- Do not provide answers.
- If the additional context does not introduce any new information, return the original questions unchanged.


QUESTIONS:
"""

In [28]:
REFINE_PROMPT_QUESTIONS = PromptTemplate(
    input_variables = ['existing_answer', "text"],
    template = refine_template,
)

In [29]:
from langchain_classic.chains.summarize import load_summarize_chain

In [30]:
import langchain
print(langchain.__version__)

1.3.14


In [31]:
ques_gen_chain = load_summarize_chain(llm= llm_ques_gen_pipeline,
                                      chain_type = "refine",
                                      verbose = True,
                                      question_prompt = PROMPT_QUESTIONS,
                                      refine_prompt = REFINE_PROMPT_QUESTIONS)

In [32]:
ques = ques_gen_chain.run(doc_ques_gen)
print(ques)



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert at creating high-quality questions from educational materials and documentation.

Your goal is to help students, professionals, and learners thoroughly understand the given content by generating meaningful and relevant questions.

Study the following text carefully:

----------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of p

C:\Users\shiva\AppData\Local\Temp\ipykernel_23976\3118617650.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ques = ques_gen_chain.run(doc_ques_gen)



> Finished chain.

> Finished chain.
1. In what year did leaders from 193 countries establish the Sustainable Development Goals (SDGs), and what is the target year set to fulfill them?

2. What is the specific role of the United Nations Development Programme (UNDP) in relation to the SDGs, and in approximately how many countries and territories does it operate?

3. According to the text, how many people worldwide still live on less than $1.25 a day, and what geographic population equivalent is used to illustrate this figure?

4. What proportion of the global population still goes to bed hungry every night, and what specific agricultural actions does the text suggest to help end hunger and malnutrition?

5. Despite historical progress in reducing child mortality, how many children still die every year before their fifth birthday, and what is the leading cause of death for adolescents in sub-Saharan Africa?

6. How does household wealth affect a child's likelihood of being out of school

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [34]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    doc_ans_gen,
    embeddings
)

In [35]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.3,
)

In [36]:
print(ques, "\n")

1. In what year did leaders from 193 countries establish the Sustainable Development Goals (SDGs), and what is the target year set to fulfill them?

2. What is the specific role of the United Nations Development Programme (UNDP) in relation to the SDGs, and in approximately how many countries and territories does it operate?

3. According to the text, how many people worldwide still live on less than $1.25 a day, and what geographic population equivalent is used to illustrate this figure?

4. What proportion of the global population still goes to bed hungry every night, and what specific agricultural actions does the text suggest to help end hunger and malnutrition?

5. Despite historical progress in reducing child mortality, how many children still die every year before their fifth birthday, and what is the leading cause of death for adolescents in sub-Saharan Africa?

6. How does household wealth affect a child's likelihood of being out of school, and what milestone primary education

In [46]:
ques_list = [ques]
ques_list = ques.split("\n")
ques_list


['1. In what year did leaders from 193 countries establish the Sustainable Development Goals (SDGs), and what is the target year set to fulfill them?',
 '',
 '2. What is the specific role of the United Nations Development Programme (UNDP) in relation to the SDGs, and in approximately how many countries and territories does it operate?',
 '',
 '3. According to the text, how many people worldwide still live on less than $1.25 a day, and what geographic population equivalent is used to illustrate this figure?',
 '',
 '4. What proportion of the global population still goes to bed hungry every night, and what specific agricultural actions does the text suggest to help end hunger and malnutrition?',
 '',
 '5. Despite historical progress in reducing child mortality, how many children still die every year before their fifth birthday, and what is the leading cause of death for adolescents in sub-Saharan Africa?',
 '',
 "6. How does household wealth affect a child's likelihood of being out of sc

In [37]:
from langchain_classic.chains import RetrievalQA

In [38]:
answer_generation_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

In [44]:
# Answer each question and save to a file
for question in ques_list:
    print("Question: ", question)
    answer = answer_generation_chain.run(question)
    print("Answer: ", answer)
    print("----------------------------------\\n\n")
    # Save answer to file
    with open("answers.txt", "a") as f:
        f.write("Question: " + question + "\\n")
        f.write("Answer: " + answer + "\\n")
        f.write("----------------------------------\\n\n")


Question:  1. In what year did leaders from 193 countries establish the Sustainable Development Goals (SDGs), and what is the target year set to fulfill them?

2. What is the specific role of the United Nations Development Programme (UNDP) in relation to the SDGs, and in approximately how many countries and territories does it operate?

3. According to the text, how many people worldwide still live on less than $1.25 a day, and what geographic population equivalent is used to illustrate this figure?

4. What proportion of the global population still goes to bed hungry every night, and what specific agricultural actions does the text suggest to help end hunger and malnutrition?

5. Despite historical progress in reducing child mortality, how many children still die every year before their fifth birthday, and what is the leading cause of death for adolescents in sub-Saharan Africa?

6. How does household wealth affect a child's likelihood of being out of school, and what milestone primar